# EDA bộ dữ liệu BI10 Vòng 2

Notebook này xem qua cả 5 bảng dữ liệu của đề (CUSTOMER, CONTRACT, LEADS, PAST_BEHAVIOR, CALL) trước khi nhóm bắt đầu chấm điểm. Mục tiêu là trả lời ba câu hỏi: mỗi bảng chứa gì và mỗi dòng đại diện cho cái gì; dữ liệu có lỗi gì mà cả nhóm phải xử lý giống nhau; và những cột nào có thể dùng làm tín hiệu cho Financial Health Score và Engagement Score. Phần cuối notebook tổng hợp các ý chính.

Dữ liệu được đọc bằng DuckDB, một thư viện chạy câu lệnh SQL ngay trong Python. Lý do là file CONTRACT nặng hơn 1 GB, đọc bằng pandas sẽ chậm và dễ hết bộ nhớ. Lần chạy đầu mất khoảng 1–2 phút để nạp 5 file CSV vào file `bi10_eda.duckdb` nằm cạnh notebook; các lần chạy sau dùng lại file này.

Cài thư viện: `pip install duckdb pandas matplotlib`. Notebook đặt trong thư mục `BI10_Round2`, cùng cấp với thư mục `BI10_ROUND02_DATASET`.

In [1]:
import duckdb, pandas as pd, matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

DATA_DIR = 'BI10_ROUND02_DATASET'
con = duckdb.connect('bi10_eda.duckdb')
con.execute("SET memory_limit='2GB'")
con.execute("SET preserve_insertion_order=false")

def q(sql):
    return con.sql(sql).df()

TABLES = ['CUSTOMER', 'CONTRACT', 'LEADS', 'PAST_BEHAVIOR', 'CALL']
existing = set(q("select table_name from information_schema.tables")['table_name'])
for t in TABLES:
    if t not in existing:
        con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv('{DATA_DIR}/{t}.csv', header=true, sample_size=-1)")
print('Đã nạp:', TABLES)

IOException: IO Error: No files found that match the pattern "BI10_ROUND02_DATASET/CUSTOMER.csv"

## 1. Tổng quan 5 bảng

Trước tiên cần biết mỗi bảng có bao nhiêu dòng, bao nhiêu khách và trải trong khoảng thời gian nào. Mỗi bảng có một cột thời gian chính: ngày nộp hồ sơ với CONTRACT, ngày tạo lead với LEADS, ngày xảy ra sự kiện với PAST_BEHAVIOR, giờ bắt đầu cuộc gọi với CALL. CUSTOMER không có cột thời gian.

In [ ]:
q('''
select 'CUSTOMER' bang, count(*) so_dong, count(distinct SKP_CLIENT) so_khach, null tu_ngay, null den_ngay from CUSTOMER
union all select 'CONTRACT', count(*), count(distinct SKP_CLIENT), min(APPLY_CONTRACT_TIME)::date, max(APPLY_CONTRACT_TIME)::date from CONTRACT
union all select 'LEADS', count(*), count(distinct SKP_CLIENT), min(DTIME_CREATED)::date, max(DTIME_CREATED)::date from LEADS
union all select 'PAST_BEHAVIOR', count(*), count(distinct SKP_CLIENT), min(DTIME_EVENT)::date, max(DTIME_EVENT)::date from PAST_BEHAVIOR
union all select 'CALL', count(*), count(distinct SKP_CLIENT), min(TIME_OF_CALL_START)::date, max(TIME_OF_CALL_START)::date from CALL
''')

CUSTOMER có đúng một dòng cho mỗi khách, nên đây là bảng gốc để gắn điểm cho từng khách. CONTRACT có số khách bằng CUSTOMER, nghĩa là khách nào trong danh sách cũng đã từng nộp ít nhất một hồ sơ vay.

Ba bảng CONTRACT, LEADS, PAST_BEHAVIOR cùng dừng ở ngày 04/06/2025 (PAST_BEHAVIOR dừng ở 03/06). Riêng CALL bắt đầu từ 01/2025 và kéo dài đến 31/08/2026, tức là sau ngày cuối của các bảng khác 15 tháng; phần 6 xem kỹ điểm này. Notebook lấy **04/06/2025 làm ngày chốt** để tính số ngày kể từ lần tương tác gần nhất.

In [ ]:
SNAPSHOT = '2025-06-04'

### 1.1. Các bảng nối với nhau thế nào

Cả 5 bảng đều có cột `SKP_CLIENT`. Câu hỏi là có khách nào xuất hiện ở bảng con mà không có trong CUSTOMER không; nếu có, những khách đó sẽ bị mất khi nối.

In [ ]:
q('''
select 'CONTRACT' bang, count(distinct SKP_CLIENT) so_khach, count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CUSTOMER)) khong_co_trong_CUSTOMER from CONTRACT
union all select 'LEADS', count(distinct SKP_CLIENT), count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CUSTOMER)) from LEADS
union all select 'PAST_BEHAVIOR', count(distinct SKP_CLIENT), count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CUSTOMER)) from PAST_BEHAVIOR
union all select 'CALL', count(distinct SKP_CLIENT), count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CUSTOMER)) from CALL
''')

Không có khách nào bị lệch, nên có thể nối mọi bảng vào CUSTOMER qua `SKP_CLIENT` mà không mất dữ liệu.

Câu hỏi tiếp theo quan trọng cho Engagement Score: mỗi khách có mặt ở những bảng hoạt động nào. Ở bảng dưới, 1 là có ít nhất một dòng trong bảng đó, 0 là không có.

In [ ]:
q('''
select (SKP_CLIENT in (select SKP_CLIENT from LEADS))::int co_lead,
       (SKP_CLIENT in (select SKP_CLIENT from PAST_BEHAVIOR))::int co_behavior,
       (SKP_CLIENT in (select SKP_CLIENT from CALL))::int co_call,
       count(*) so_khach, round(100.0 * count(*) / sum(count(*)) over (), 1) pct
from CUSTOMER group by all order by so_khach desc
''')

Chỉ 19% khách có mặt ở cả ba bảng hoạt động. Một nhóm khoảng 190 nghìn khách (19%) không có lead, không có sự kiện chấm điểm và không có cuộc gọi nào; với nhóm này, Engagement chỉ có thể tính từ lịch sử hợp đồng. Điều này cần được tính đến khi thiết kế điểm: nếu cộng điểm theo số lần xuất hiện ở mỗi bảng, nhóm này sẽ tự động bị điểm 0 ở phần lớn thành phần.

## 2. CUSTOMER

Mỗi dòng là một khách với thông tin mới nhất. Câu hỏi ở đây là cột nào dùng được và cột nào gần như trống.

In [ ]:
q('''
select
  round(100 * avg((FLAG_CAR_OWNER = 'X')::int), 2) pct_car_owner_la_X,
  round(100 * avg((NAME_SALARY_FREQUENCY = 'XNA')::int), 2) pct_salary_freq_XNA,
  round(100 * avg((CNT_PERSON_DEPENDENT is null)::int), 2) pct_dependent_trong,
  round(100 * avg((CNT_CHILDREN is null)::int), 2) pct_children_trong,
  round(100 * avg((NAME_EDUCATION_TYPE = 'XNA')::int), 2) pct_education_XNA,
  round(100 * avg((ADDRESS is null)::int), 2) pct_address_trong,
  round(100 * avg((NAME_GENDER = 'XNA')::int), 2) pct_gender_XNA
from CUSTOMER
''')

Ba cột không dùng được: `FLAG_CAR_OWNER` có giá trị "X" ở 100% dòng nên không phân biệt được ai; `NAME_SALARY_FREQUENCY` là XNA ở 99%; `CNT_PERSON_DEPENDENT` trống 98,5%. Học vấn XNA 16%, địa chỉ trống 15%, số con trống 12,6%.

Số con còn có giá trị bất thường, xem phân bố dưới đây.

In [ ]:
q('''
select case when CNT_CHILDREN is null then 'trống' when CNT_CHILDREN <= 5 then CNT_CHILDREN::varchar else 'trên 5' end so_con,
       count(*) so_khach
from CUSTOMER group by 1 order by 2 desc
''')

Có 965 khách có số con trên 5, phần lớn nhiều khả năng là lỗi nhập. Nếu dùng cột này cần gom các giá trị trên 5 thành một nhóm.

### 2.1. Tuổi

Tuổi được tính tại ngày chốt bằng số ngày từ ngày sinh chia cho 365,25.

In [ ]:
age = q(f'''
select floor(date_diff('day', DATE_BIRTH, DATE '{SNAPSHOT}') / 365.25)::int tuoi from CUSTOMER
''')
print('Dưới 18 tuổi:', (age.tuoi < 18).sum(), '| Trên 80 tuổi:', (age.tuoi > 80).sum())
print(age.tuoi.describe().round(1))
age[(age.tuoi >= 18) & (age.tuoi <= 80)].tuoi.plot.hist(bins=50, figsize=(8, 3), title='Phân bố tuổi khách tại 04/06/2025')
plt.xlabel('Tuổi'); plt.ylabel('Số khách'); plt.show()

Một nửa số khách ở độ tuổi 28–43, tuổi trung vị là 35. Có 25 khách có tuổi bất thường, thấp nhất 1 tuổi và cao nhất 123 tuổi, có thể loại hoặc gán trống.

### 2.2. Tỉnh

Địa chỉ được ghi theo dạng "phường, quận, tỉnh", nên tỉnh là phần sau dấu phẩy cuối cùng.

In [ ]:
prov = q('''
select trim(split_part(ADDRESS, ',', -1)) tinh, count(*) so_khach from CUSTOMER group by 1 order by 2 desc
''')
print('Số cách ghi tỉnh khác nhau:', prov.tinh.notna().sum())
prov.head(12)

Có 68 cách ghi tên tỉnh, nhiều hơn 63 tỉnh, nên cần chuẩn hóa tên trước khi dùng. Địa chỉ dùng tên tỉnh trước khi sáp nhập năm 2025: Long An, Kien Giang, Binh Duong vẫn là tỉnh riêng. TP.HCM nhiều khách nhất với khoảng 100 nghìn; các tỉnh phía Nam chiếm phần lớn danh sách đầu.

### 2.3. Số phiên dùng app năm 2025

`AVG_SESSION_PER_WEEK_2025` là số lần dùng ứng dụng trung bình mỗi tuần trong năm 2025, có thể là tín hiệu tương tác trực tiếp.

In [ ]:
q('''
select round(100 * avg((AVG_SESSION_PER_WEEK_2025 = 0)::int), 1) pct_bang_0,
       count(*) filter (where AVG_SESSION_PER_WEEK_2025 > 0) so_khach_co_dung_app,
       quantile_cont(AVG_SESSION_PER_WEEK_2025, 0.5) filter (where AVG_SESSION_PER_WEEK_2025 > 0) trung_vi_khi_co_dung,
       quantile_cont(AVG_SESSION_PER_WEEK_2025, 0.9) filter (where AVG_SESSION_PER_WEEK_2025 > 0) p90_khi_co_dung,
       max(AVG_SESSION_PER_WEEK_2025) cao_nhat
from CUSTOMER
''')

93% khách có giá trị 0, nên cột này chỉ phân biệt được khoảng 70 nghìn khách có dùng app. Với Engagement Score, nên dùng cột này dưới dạng "có dùng app hay không" kèm mức độ, thay vì để nó chi phối điểm.

## 3. CONTRACT

Mỗi dòng theo mô tả của đề là một hồ sơ vay. Đây là bảng quan trọng nhất cho Financial Health Score vì chứa trạng thái hợp đồng và các cờ quá hạn.

### 3.1. Trùng dòng

Câu hỏi đầu tiên là `SKP_CREDIT_CASE` có thật sự là duy nhất không.

In [ ]:
q('''
select so_dong_moi_hd, count(*) so_hop_dong
from (select SKP_CREDIT_CASE, count(*) so_dong_moi_hd from CONTRACT group by 1)
group by 1 order by 1
''')

Gần 2 triệu hợp đồng xuất hiện 2 lần. Xem cụ thể một hợp đồng để biết hai dòng khác nhau ở đâu.

In [ ]:
q('''
select SKP_CREDIT_CASE, SKP_CLIENT, NAME_EDUCATION_TYPE, CNT_CHILDREN, AMT_INCOME_MAIN, NAME_INCOME_TYPE, CODE_PROFESSION,
       NAME_CREDIT_STATUS, PRODUCT, AMT_CREDIT, PAYMENT_NUM, ANNUITY, APPLY_CONTRACT_TIME, FIRST_DUE, FOURTH_DUE, FLAG_INS
from CONTRACT where SKP_CREDIT_CASE in (46021692, 46058113) order by 1
''')

Hai dòng của cùng hợp đồng giống nhau ở các trường nghiệp vụ, chỉ khác ở phần thông tin cá nhân: một dòng có học vấn, thu nhập, nghề nghiệp đầy đủ, dòng kia là XNA hoặc 0. Kiểm tra trên toàn bộ các hợp đồng bị trùng xem các trường nghiệp vụ có bao giờ khác nhau không.

In [ ]:
q('''
select count(*) so_hd_trung, sum(a::int) khac_trang_thai, sum(b::int) khac_san_pham, sum(c::int) khac_so_tien,
  sum(d::int) khac_ngay_ky, sum(e::int) khac_co_qua_han, sum(f::int) khac_bao_hiem, sum(g::int) khac_hoc_van
from (
  select SKP_CREDIT_CASE,
    min(NAME_CREDIT_STATUS) <> max(NAME_CREDIT_STATUS) a,
    min(PRODUCT) <> max(PRODUCT) b,
    min(AMT_CREDIT) <> max(AMT_CREDIT) c,
    min(SIGN_CONTRACT_TIME) <> max(SIGN_CONTRACT_TIME) d,
    min(FIRST_DUE+SECOND_DUE*2+THIRD_DUE*4+FOURTH_DUE*8) <> max(FIRST_DUE+SECOND_DUE*2+THIRD_DUE*4+FOURTH_DUE*8) e,
    min(FLAG_INS) <> max(FLAG_INS) f,
    min(NAME_EDUCATION_TYPE) <> max(NAME_EDUCATION_TYPE) g
  from CONTRACT group by 1 having count(*) > 1)
''')

Không có hợp đồng nào khác nhau ở trạng thái, sản phẩm, số tiền, ngày ký, cờ quá hạn hay bảo hiểm; khác biệt chỉ nằm ở thông tin cá nhân. Cách xử lý: mỗi `SKP_CREDIT_CASE` giữ một dòng, ưu tiên dòng có học vấn, nghề nghiệp, loại thu nhập không phải XNA. Từ đây notebook dùng bảng đã khử trùng `CONTRACT_CLEAN`. Nếu đếm trên bảng gốc, mọi chỉ số theo hợp đồng sẽ bị nhân lên khoảng 1,5 lần.

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE CONTRACT_CLEAN AS
SELECT *,
  NAME_CREDIT_STATUS in ('Active','Finished','Paid off','Signed','Written off') AS IS_SIGNED,
  greatest(FIRST_DUE, SECOND_DUE, THIRD_DUE, FOURTH_DUE) AS ANY_DUE
FROM CONTRACT
QUALIFY row_number() OVER (PARTITION BY SKP_CREDIT_CASE
        ORDER BY (NAME_EDUCATION_TYPE='XNA' AND NAME_INCOME_TYPE='XNA' AND CODE_PROFESSION='XNA'), AMT_INCOME_MAIN DESC) = 1
''')
q('select count(*) so_dong, count(distinct SKP_CREDIT_CASE) so_hop_dong, count(distinct SKP_CLIENT) so_khach from CONTRACT_CLEAN')

### 3.2. Trạng thái hợp đồng

Theo mô tả của đề, hợp đồng đã ký là Active, Finished, Paid off, Signed, Written off; các trạng thái còn lại là chưa ký. Cột `IS_SIGNED` vừa tạo theo đúng quy ước này.

In [ ]:
q('''
select NAME_CREDIT_STATUS, IS_SIGNED, count(*) so_hd, round(100.0 * count(*) / sum(count(*)) over (), 2) pct,
       round(100 * avg((APPROVE_CONTRACT_TIME is not null)::int), 1) pct_co_ngay_duyet,
       round(100 * avg((SIGN_CONTRACT_TIME is not null)::int), 1) pct_co_ngay_ky
from CONTRACT_CLEAN group by 1, 2 order by so_hd desc
''')

Gần một nửa hợp đồng đã kết thúc (Finished), 20% đang trả (Active), 21% bị từ chối và 11% bị hủy. Written off, tức nợ không thu hồi được và phải xóa sổ, chỉ chiếm 0,34% nhưng là tín hiệu tiêu cực rõ nhất.

Khoảng 10% hợp đồng Finished không có ngày ký. Bảng dưới cho thấy lý do: ngày ký chỉ được ghi từ năm 2017.

In [ ]:
q('''
select year(APPLY_CONTRACT_TIME) nam, count(*) so_hd_da_ky, round(100 * avg((SIGN_CONTRACT_TIME is null)::int), 1) pct_thieu_ngay_ky
from CONTRACT_CLEAN where IS_SIGNED group by 1 order by 1
''')

Hợp đồng nộp trước 2017 hoàn toàn không có ngày ký, năm 2017 thiếu 22%. Vì vậy phải xác định "đã ký" theo trạng thái, không theo việc có ngày ký hay không.

### 3.3. Sản phẩm và số tiền

Đề mô tả 5 sản phẩm: CD là vay trả góp điện máy, TW là vay mua xe máy, BNPL là tài khoản trả sau, CC là thẻ tín dụng, CL là vay tiền mặt.

In [ ]:
q('''
select PRODUCT, count(*) so_hd, round(100.0 * count(*) / sum(count(*)) over (), 1) pct,
  median(AMT_CREDIT) trung_vi_amt_credit, median(AMT_BILLING) trung_vi_amt_billing,
  median(PAYMENT_NUM) trung_vi_ky_han, median(ANNUITY) trung_vi_tra_thang,
  round(100 * avg((INIT_PAY > 0)::int), 1) pct_co_tra_truoc,
  round(100 * avg((TRANSAC is not null)::int), 1) pct_co_TRANSAC,
  round(100 * avg(FLAG_INS), 1) pct_bao_hiem
from CONTRACT_CLEAN group by 1 order by so_hd desc
''')

Có bốn điểm cần lưu ý:

- CD chiếm 72% số hợp đồng nhưng khoản vay nhỏ: trung vị khoảng 5,6 triệu, kỳ hạn 6 tháng. CL và TW có khoản vay lớn nhất, trung vị khoảng 22 triệu.
- BNPL có `AMT_CREDIT` bằng 0, nên số tiền của BNPL phải lấy từ `AMT_BILLING`, là số tiền được tính trong kỳ sao kê đầu tiên. CC có cả hai cột.
- `TRANSAC` được đề mô tả là mã kênh, nhưng dữ liệu thực tế là ngày, và chỉ có ở BNPL và CC.
- CW chiếm 2,6% hợp đồng nhưng không có trong mô tả của đề. Có 28 nghìn hợp đồng không có sản phẩm, và 1 hợp đồng IN.

Kiểm tra nhóm không có sản phẩm:

In [ ]:
q('''
select NAME_CREDIT_STATUS, count(*) so_hd, max(AMT_CREDIT) amt_lon_nhat
from CONTRACT_CLEAN where PRODUCT is null group by 1 order by 2 desc
''')

Cả 28 nghìn hợp đồng không có sản phẩm đều là hồ sơ bị từ chối hoặc bị hủy, số tiền bằng 0, nên không ảnh hưởng đến phân tích trên hợp đồng đã ký.

### 3.4. Số hồ sơ theo năm

In [ ]:
by_year = q('''
select year(APPLY_CONTRACT_TIME) nam, PRODUCT, count(*) so_hd from CONTRACT_CLEAN
where year(APPLY_CONTRACT_TIME) >= 2016 group by 1, 2
''').pivot(index='nam', columns='PRODUCT', values='so_hd').fillna(0).astype(int)
display(by_year)
by_year.plot.bar(stacked=True, figsize=(9, 3.5), title='Số hồ sơ vay theo năm nộp và sản phẩm (2025 chỉ đến 04/06)')
plt.ylabel('Số hợp đồng'); plt.xlabel('Năm'); plt.show()

Số hồ sơ tăng mạnh năm 2024, gấp khoảng 2,7 lần năm 2023, và chỉ 5 tháng đầu 2025 đã bằng khoảng 80% cả năm 2024. BNPL bắt đầu từ giữa 2022. Như vậy phần lớn hoạt động của khách nằm trong khoảng 2024–2025, trùng với giai đoạn có dữ liệu LEADS, PAST_BEHAVIOR và CALL.

### 3.5. Cờ quá hạn

`FIRST_DUE` đến `FOURTH_DUE` có giá trị 0 hoặc 1. Câu hỏi là các cờ này có cộng dồn không, tức là đã quá hạn kỳ 4 thì có chắc đã quá hạn kỳ 1 không.

In [ ]:
q('''
select count(*) so_hd_da_ky,
  round(100 * avg(FIRST_DUE), 2) pct_qh_ky1, round(100 * avg(SECOND_DUE), 2) pct_qh_ky2,
  round(100 * avg(THIRD_DUE), 2) pct_qh_ky3, round(100 * avg(FOURTH_DUE), 2) pct_qh_ky4,
  round(100 * avg(ANY_DUE), 2) pct_qh_it_nhat_1_ky,
  count(*) filter (where FOURTH_DUE = 1 and FIRST_DUE = 0) qh_ky4_nhung_khong_qh_ky1
from CONTRACT_CLEAN where IS_SIGNED
''')

Tỷ lệ tăng dần từ kỳ 1 đến kỳ 4 và có hàng chục nghìn hợp đồng quá hạn kỳ 4 mà không quá hạn kỳ 1, nên các cờ không cộng dồn. Cách hiểu hợp lý là mỗi cờ cho biết khách có trễ hạn ở kỳ trả thứ 1, 2, 3, 4 hay không. Trong hợp đồng đã ký, khoảng 2% có ít nhất một kỳ trễ.

Hợp đồng mới ký chưa đến kỳ thứ 4 thì cờ kỳ 4 chắc chắn bằng 0. Bảng theo năm nộp dưới đây kiểm tra điều đó.

In [ ]:
q('''
select year(APPLY_CONTRACT_TIME) nam, count(*) so_hd_da_ky,
  round(100 * avg(FIRST_DUE), 2) pct_qh_ky1, round(100 * avg(FOURTH_DUE), 2) pct_qh_ky4, round(100 * avg(ANY_DUE), 2) pct_qh_bat_ky
from CONTRACT_CLEAN where IS_SIGNED and year(APPLY_CONTRACT_TIME) >= 2017 group by 1 order by 1
''')

Kết quả ngược với dự đoán: hợp đồng nộp năm 2024 và 2025 có tỷ lệ trễ kỳ 1 khoảng 0,7%, cao gấp 2–3 lần các năm 2017–2023 (0,2–0,5%), dù phần lớn hợp đồng 2025 chưa đến kỳ 4. Một nguyên nhân có thể là số hồ sơ tăng mạnh trong 2024 kéo theo nhiều khách mới hơn; điều này cần kiểm tra thêm. Năm 2021 cũng cao bất thường (3,3% trễ ít nhất một kỳ), trùng giai đoạn dịch COVID-19. Vì cờ kỳ 4 của hợp đồng 2025 chưa phản ánh đầy đủ, khi so sánh khách với nhau nên dùng cờ kỳ 1 hoặc chỉ tính các kỳ đã đến hạn.

Tỷ lệ trễ hạn theo sản phẩm và theo trạng thái:

In [ ]:
display(q('''
select PRODUCT, count(*) so_hd_da_ky, round(100 * avg(ANY_DUE), 2) pct_qh_bat_ky, round(100 * avg(FOURTH_DUE), 2) pct_qh_ky4,
       round(100 * avg((NAME_CREDIT_STATUS = 'Written off')::int), 2) pct_written_off
from CONTRACT_CLEAN where IS_SIGNED group by 1 order by 2 desc
'''))
q('''
select NAME_CREDIT_STATUS, count(*) so_hd, round(100 * avg(ANY_DUE), 2) pct_qh_bat_ky
from CONTRACT_CLEAN where IS_SIGNED group by 1 order by 2 desc
''')

Tỷ lệ trễ hạn khác nhau rõ giữa các sản phẩm: TW 4,2% và CW 2,8%, trong khi CL chỉ 1,3%. CW còn có tỷ lệ Written off cao nhất, gần 3%. Vì vậy khi chấm Financial Health cần so khách với mặt bằng của đúng sản phẩm đó.

Hợp đồng Written off có 53,5% trễ ít nhất một kỳ đầu, so với 1,35% của hợp đồng Finished. Như vậy việc trễ ở 4 kỳ đầu thường xảy ra trước khi khách mất khả năng trả nợ, và đây là bằng chứng để giải thích vì sao cờ quá hạn được chọn làm tín hiệu chính.

Paid off có tới 82% trễ hạn, nên Paid off không phải là trả xong sớm như tên gọi gợi ý; có thể là tất toán sau khi đã trễ hạn. Không nên coi Paid off là tín hiệu tốt khi chưa rõ nghĩa.

### 3.6. Nhìn theo từng khách

Gom hợp đồng về mức khách để xem một khách điển hình có bao nhiêu hợp đồng.

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE CLIENT_CONTRACT AS
SELECT SKP_CLIENT,
  count(*) n_hd, sum(IS_SIGNED::int) n_da_ky,
  sum((NAME_CREDIT_STATUS = 'Rejected')::int) n_tu_choi,
  sum((NAME_CREDIT_STATUS = 'Active')::int) n_active,
  count(distinct PRODUCT) filter (where IS_SIGNED) n_san_pham_da_ky,
  max(case when IS_SIGNED then ANY_DUE else 0 end) co_qua_han,
  max((NAME_CREDIT_STATUS = 'Written off')::int) co_written_off,
  max(APPLY_CONTRACT_TIME) lan_nop_gan_nhat
FROM CONTRACT_CLEAN GROUP BY 1
''')
q('''
select count(*) so_khach,
  median(n_hd) trung_vi_so_hd, quantile_cont(n_hd, 0.9) p90_so_hd, max(n_hd) max_so_hd,
  count(*) filter (where n_da_ky = 0) khach_chua_tung_ky,
  count(*) filter (where n_active > 0) khach_dang_co_hd_active,
  count(*) filter (where co_qua_han = 1) khach_tung_qua_han,
  count(*) filter (where co_written_off = 1) khach_co_written_off,
  count(*) filter (where n_san_pham_da_ky >= 2) khach_dung_tu_2_sp
from CLIENT_CONTRACT
''')

Khoảng 182 nghìn khách (18%) chưa từng ký hợp đồng nào, chỉ có hồ sơ bị từ chối hoặc bị hủy. Với nhóm này không có lịch sử trả nợ để chấm Financial Health; nhóm cần quyết định gắn nhãn riêng hay chấm bằng tín hiệu khác. Khoảng 60% khách đang có ít nhất một hợp đồng Active, tức đang có khoản phải trả hằng tháng. 5,1% khách từng trễ hạn ít nhất một lần và 1,3% có hợp đồng Written off. 37% khách đã ký từ 2 sản phẩm khác nhau trở lên, đây là tín hiệu cho phần tương tác sản phẩm của Engagement. Có khách tới 1.794 hồ sơ, cần xem là ngoại lai khi đếm số hồ sơ.

Phân bố số ngày từ lần nộp hồ sơ gần nhất đến ngày chốt:

In [ ]:
q(f'''
select case when d <= 30 then '1. trong 30 ngày' when d <= 90 then '2. 31-90 ngày' when d <= 180 then '3. 91-180 ngày'
            when d <= 365 then '4. 181-365 ngày' when d <= 730 then '5. 1-2 năm' else '6. trên 2 năm' end nhom, count(*) so_khach,
       round(100.0 * count(*) / sum(count(*)) over (), 1) pct
from (select date_diff('day', lan_nop_gan_nhat, TIMESTAMP '{SNAPSHOT}') d from CLIENT_CONTRACT) group by 1 order by 1
''')

Không có khách nào có lần nộp hồ sơ gần nhất cách ngày chốt quá một năm, và 81,5% khách có lần nộp gần nhất cách ngày chốt 91–180 ngày. Như vậy danh sách khách được chọn từ những người đã nộp hồ sơ trong khoảng 12/2024 đến 03/2025. Recency tính theo hồ sơ vay gần như không phân biệt được khách; khi tính Engagement cần dùng thêm recency của lead, sự kiện chấm điểm và cuộc gọi.

### 3.7. Thu nhập và thông tin nghề nghiệp

Đề lưu ý không dùng thông tin nhân khẩu học để đánh giá rủi ro khi chưa đủ bằng chứng. Trước khi quyết định, cần biết các cột này đầy đủ đến đâu.

In [ ]:
display(q('''
select year(APPLY_CONTRACT_TIME) nam, count(*) so_hd,
  round(100 * avg((AMT_INCOME_MAIN = 0)::int), 1) pct_thu_nhap_0,
  round(100 * avg((NAME_INCOME_TYPE = 'XNA')::int), 1) pct_loai_thu_nhap_XNA
from CONTRACT_CLEAN where year(APPLY_CONTRACT_TIME) >= 2017 group by 1 order by 1
'''))
q('''
select quantile_cont(AMT_INCOME_MAIN, [0.01, 0.25, 0.5, 0.75, 0.99]) phan_vi_thu_nhap, max(AMT_INCOME_MAIN) cao_nhat
from CONTRACT_CLEAN where AMT_INCOME_MAIN > 0
''')

Sau khi khử trùng, gần như mọi hợp đồng từ 2018 đều có thu nhập lớn hơn 0, vì dòng bị bỏ chính là dòng thiếu thông tin. Tuy vậy loại thu nhập vẫn là XNA ở 20–28% hợp đồng từ 2019 trở đi và có xu hướng tăng, nên việc thiếu dữ liệu không xảy ra ngẫu nhiên mà phụ thuộc vào giai đoạn thu thập. Thu nhập cao nhất lên tới 9 tỷ đồng mỗi tháng, cần cắt ngoại lai nếu dùng. Bảng dưới chỉ để tham khảo, không phải bằng chứng nhân quả:

In [ ]:
q('''
select NAME_INCOME_TYPE, count(*) so_hd_da_ky, round(100 * avg(ANY_DUE), 2) pct_qh_bat_ky
from CONTRACT_CLEAN where IS_SIGNED group by 1 order by 2 desc
''')

Tỷ lệ trễ hạn giữa các loại thu nhập có thông tin chỉ dao động từ 1,5% đến 2,7%, ví dụ người làm công 1,8% và người tự kinh doanh 2,2%. Chênh lệch này nhỏ hơn nhiều so với chênh lệch giữa các sản phẩm (1,3% ở CL đến 4,2% ở TW), và các nhóm như Unemployed, Retired quá ít hợp đồng để kết luận. Đây là lý do hợp lý để không đưa thông tin nhân khẩu học vào điểm. Nhóm XNA có tỷ lệ cao nhất (4,2%) nhưng chỉ có 18 nghìn hợp đồng.

## 4. LEADS

Mỗi dòng là một lead, tức một lần công ty xác định khách có thể được chào một sản phẩm. `DESC_FIN_REASON` là kết quả cuối cùng của lead, trong đó Deal nghĩa là chốt được.

### 4.1. Trùng dòng

In [ ]:
q('select (select count(*) from LEADS) so_dong, (select count(*) from (select distinct * from LEADS)) so_dong_khong_trung')

Có khoảng 40 nghìn dòng trùng hoàn toàn ở mọi cột. Notebook dùng bảng `LEADS_CLEAN` đã bỏ các dòng này.

In [ ]:
con.execute('CREATE OR REPLACE TABLE LEADS_CLEAN AS SELECT DISTINCT * FROM LEADS')
q('''
select DESC_FIN_REASON, count(*) so_lead, round(100.0 * count(*) / sum(count(*)) over (), 2) pct
from LEADS_CLEAN group by 1 order by 2 desc
''')

58% lead kết thúc ở Not Interest và 22% hết hạn mà chưa xử lý. Chỉ 2,7% lead thành Deal.

### 4.2. Nguồn lead

Đề mô tả `DATE_ACCEPTED = 3000-01-01` là lead từ SOB, nơi nhân viên hỗ trợ trực tiếp mà không bấm nhận lead. Bảng dưới kiểm tra mô tả này và so sánh tỷ lệ Deal theo nguồn.

In [ ]:
src = q('''
select LEAD_SOURCE, count(*) so_lead,
  round(100 * avg((DATE_ACCEPTED = '3000-01-01')::int), 1) pct_ngay_3000,
  round(100 * avg(FLAG_ASSIGNED), 1) pct_duoc_giao,
  round(100 * avg((DESC_FIN_REASON = 'Deal')::int), 2) pct_deal,
  min(DTIME_CREATED)::date tu_ngay
from LEADS_CLEAN group by 1 order by so_lead desc
''')
display(src)
src.sort_values('pct_deal').plot.barh(x='LEAD_SOURCE', y='pct_deal', legend=False, figsize=(7, 3), title='Tỷ lệ lead thành Deal theo nguồn (%)')
plt.xlabel('% Deal'); plt.ylabel(''); plt.show()

Hai điểm rút ra:

- Giá trị 3000-01-01 xuất hiện ở mọi nguồn, từ 12% đến 52%, không chỉ ở SOB như mô tả. Nên hiểu đây là "chưa có ngày nhận lead" và không tính khoảng thời gian từ lúc tạo đến lúc nhận với các dòng này.
- RA và SA chiếm 94% số lead nhưng tỷ lệ Deal chỉ 0,2% và 1,9%. Các nguồn mà khách chủ động hoặc có người tư vấn trực tiếp như SOB, Landing Page, TLS có tỷ lệ Deal 37–40%. Đây là bằng chứng cho phần Next Best Action: kênh tiếp cận ảnh hưởng mạnh đến khả năng chốt. Tuy vậy, khác biệt này có thể một phần do khách tự tìm đến các kênh đó vì đã có nhu cầu, nên chưa thể kết luận là đổi kênh sẽ tăng tỷ lệ chốt.

### 4.3. Sản phẩm trong lead

In [ ]:
q('''
select CODE_PRODUCT_TYPE, count(*) so_lead, round(100 * avg((DESC_FIN_REASON = 'Deal')::int), 2) pct_deal, median(MAX_OFFER) trung_vi_han_muc
from LEADS_CLEAN group by 1 order by 2 desc
''')

Mã sản phẩm ở LEADS khác với CONTRACT (BNP, CLX, CCX, SCW, VCC…), phần 7 tổng hợp lại. BNP có 734 nghìn lead nhưng không có Deal nào, có thể kết quả của BNPL được ghi nhận ở nơi khác.

### 4.4. Deal có thật sự dẫn đến hợp đồng không

Nếu dùng Deal làm kết quả kinh doanh để kiểm chứng Engagement Score, cần biết Deal có khớp với việc khách nộp và ký hợp đồng không. Cách kiểm tra: với mỗi lead tạo từ 01/2024 đến 04/05/2025, xem khách có nộp hồ sơ trong 30 ngày sau đó không. Mốc 04/05/2025 để lead nào cũng có đủ 30 ngày theo dõi trước ngày chốt.

In [ ]:
q('''
with l as (select SKP_CLIENT, DTIME_CREATED, DESC_FIN_REASON = 'Deal' la_deal from LEADS_CLEAN
           where DTIME_CREATED >= '2024-01-01' and DTIME_CREATED < '2025-05-05'),
c as (select SKP_CLIENT, APPLY_CONTRACT_TIME t, IS_SIGNED from CONTRACT_CLEAN where APPLY_CONTRACT_TIME >= '2024-01-01'),
m as (select l.*, c.t, c.IS_SIGNED from l asof left join c on l.SKP_CLIENT = c.SKP_CLIENT and c.t >= l.DTIME_CREATED)
select la_deal, count(*) so_lead,
  round(100 * avg((t is not null and t <= DTIME_CREATED + interval 30 day)::int), 1) pct_nop_hs_trong_30_ngay,
  round(100 * avg((t is not null and t <= DTIME_CREATED + interval 30 day and IS_SIGNED)::int), 1) pct_hs_dau_tien_da_ky
from m group by 1 order by 1
''')

75% lead Deal có hồ sơ nộp trong 30 ngày, so với 21% của lead không phải Deal. Deal đúng là phản ánh việc khách đi tiếp đến hợp đồng, nên có thể dùng Deal, hoặc việc nộp hồ sơ mới, làm kết quả để kiểm chứng Engagement Score. Phép nối ở trên dùng ASOF JOIN, tức là với mỗi lead chỉ lấy hồ sơ đầu tiên nộp sau ngày tạo lead.

### 4.5. Số lead theo năm và theo khách

In [ ]:
display(q('select year(DTIME_CREATED) nam, count(*) so_lead, count(distinct SKP_CLIENT) so_khach from LEADS_CLEAN group by 1 order by 1'))
q('''
select median(n) trung_vi_lead_moi_khach, quantile_cont(n, 0.9) p90, max(n) cao_nhat
from (select SKP_CLIENT, count(*) n from LEADS_CLEAN group by 1)
''')

## 5. PAST_BEHAVIOR

Mỗi dòng là một sự kiện: khách được chấm điểm tín dụng qua một kênh (app, nhân viên bán hàng SA, web, telesales TLS, website đối tác) hoặc nộp hồ sơ qua các quy trình như SOB, 0BOD, đến trực tiếp cửa hàng.

In [ ]:
q('''
select ACTION, count(*) so_su_kien, round(100.0 * count(*) / sum(count(*)) over (), 2) pct, count(distinct SKP_CLIENT) so_khach
from PAST_BEHAVIOR group by 1 order by 2 desc
''')

Phần lớn là sự kiện chấm điểm, trong đó qua app chiếm 37%. Các quy trình nộp hồ sơ SOB, 0BOD, Walk-in chỉ khoảng 1%. Có vài dòng gộp hai kênh như "SA, TLS SCORING", cần tách hoặc gán vào một kênh. Không có dòng trùng.

Sản phẩm trong bảng này cũng dùng mã riêng:

In [ ]:
q('select PRODUCT_CODE, count(*) so_su_kien from PAST_BEHAVIOR group by 1 order by 2 desc')

### 5.1. Số sự kiện theo tháng và theo khách

In [ ]:
m = q("select date_trunc('month', DTIME_EVENT)::date thang, count(*) so_su_kien from PAST_BEHAVIOR group by 1 order by 1")
m.plot(x='thang', y='so_su_kien', legend=False, figsize=(9, 3), title='Số sự kiện chấm điểm và nộp hồ sơ theo tháng')
plt.ylabel('Số sự kiện'); plt.xlabel(''); plt.show()
q('''
select median(n) trung_vi_su_kien_moi_khach, quantile_cont(n, 0.9) p90, max(n) cao_nhat
from (select SKP_CLIENT, count(*) n from PAST_BEHAVIOR group by 1)
''')

Số sự kiện tăng mạnh từ 2023 sang 2024, cùng xu hướng với số hồ sơ vay. Một khách trung vị có 5 sự kiện, nhưng có khách tới 1.778 sự kiện, cần xem là ngoại lai khi đếm tần suất.

### 5.2. Sau khi được chấm điểm, khách có nộp hồ sơ không

Câu hỏi này cho biết sự kiện chấm điểm có phải là tín hiệu tương tác có giá trị không, và kênh nào đi tiếp đến hồ sơ nhiều nhất. Cách tính giống phần 4.4: với mỗi sự kiện từ 01/2024 đến 04/05/2025, xem khách có nộp hồ sơ trong 30 ngày sau đó không.

In [ ]:
q('''
with e as (select SKP_CLIENT, DTIME_EVENT, ACTION from PAST_BEHAVIOR where DTIME_EVENT >= '2024-01-01' and DTIME_EVENT < '2025-05-05'),
c as (select SKP_CLIENT, APPLY_CONTRACT_TIME t from CONTRACT_CLEAN where APPLY_CONTRACT_TIME >= '2024-01-01')
select ACTION, count(*) so_su_kien, round(100 * avg((c.t is not null and c.t <= e.DTIME_EVENT + interval 30 day)::int), 1) pct_nop_hs_trong_30_ngay
from e asof left join c on e.SKP_CLIENT = c.SKP_CLIENT and c.t >= e.DTIME_EVENT
group by 1 having count(*) >= 100 order by 2 desc
''')

Tỷ lệ đi tiếp đến hồ sơ khác nhau rõ giữa các kênh. Chấm điểm qua nhân viên bán hàng (SA) dẫn đến hồ sơ trong 30 ngày ở khoảng 70% trường hợp, trong khi chấm điểm qua app chỉ khoảng 33% và qua website đối tác khoảng 23%. Các quy trình SOB gần như luôn đi tiếp đến hồ sơ vì bản thân chúng là bước nộp hồ sơ. Như vậy, khi tính Engagement, một lần chấm điểm qua app không nên có trọng số bằng một lần chấm điểm có nhân viên đi kèm.

## 6. CALL

Mỗi dòng là một cuộc gọi giữa tổng đài và khách.

### 6.1. Thời gian của bảng CALL

In [ ]:
cm = q("select date_trunc('month', TIME_OF_CALL_START)::date thang, count(*) so_cuoc_goi, count(distinct SKP_CLIENT) so_khach from CALL group by 1 order by 1")
display(cm)
colors = ['#4c72b0' if str(t) <= SNAPSHOT else '#c44e52' for t in cm.thang]
cm.plot.bar(x='thang', y='so_cuoc_goi', color=colors, legend=False, figsize=(10, 3), title='Số cuộc gọi theo tháng (đỏ: sau ngày chốt 04/06/2025)')
plt.ylabel('Số cuộc gọi'); plt.xlabel(''); plt.show()
q(f"select round(100 * avg((TIME_OF_CALL_START > TIMESTAMP '{SNAPSHOT}')::int), 1) pct_cuoc_goi_sau_ngay_chot from CALL")

Hai điểm bất thường: 75% cuộc gọi diễn ra sau ngày cuối của mọi bảng khác, và số cuộc gọi mỗi tháng gần như bằng nhau, khoảng 200 nghìn (tháng 2 ít hơn chỉ vì ít ngày hơn). Dữ liệu thật thường dao động theo chiến dịch và mùa vụ, nên nhiều khả năng ngày của bảng này đã bị dịch hoặc được tạo lại. Nhóm cần chọn một trong hai cách và ghi rõ trong slide dữ liệu:

- Chỉ dùng cuộc gọi trước ngày chốt: thời gian nhất quán, nhưng mất 75% dữ liệu.
- Dùng toàn bộ như thông tin mô tả mức độ liên lạc của khách: đủ dữ liệu hơn, nhưng không dùng được để kiểm chứng theo thời gian với hợp đồng.

### 6.2. CALL_ID bị trùng

In [ ]:
display(q('''
select count(*) so_dong, count(distinct CALL_ID) so_call_id,
       (select count(*) from (select * from CALL group by all having count(*) > 1)) nhom_dong_trung_hoan_toan
from CALL'''))
q('''
select * from CALL where CALL_ID in (select CALL_ID from CALL group by 1 having count(*) > 1 limit 3) order by CALL_ID
''')

Khoảng 12,5 nghìn `CALL_ID` xuất hiện hai lần nhưng không có dòng nào trùng hoàn toàn. Ví dụ ở trên cho thấy hai dòng cùng `CALL_ID` là hai cuộc gọi ở hai ngày khác nhau, nội dung cũng khác nhau, nên `CALL_ID` bị dùng lại chứ không phải dòng lặp. Mỗi dòng nên được tính là một cuộc gọi.

### 6.3. Kết quả cuộc gọi

In [ ]:
r = q('''
select CALL_RESULT_SYSTEM, count(*) so_cuoc_goi, round(100.0 * count(*) / sum(count(*)) over (), 2) pct,
       round(avg(DURATIONS), 1) tb_thoi_luong_giay, round(100 * avg((DURATIONS > 0)::int), 1) pct_thoi_luong_lon_hon_0
from CALL group by 1 order by 2 desc
''')
display(r)
q('''
select CALL_TYPE, OFF_BY, count(*) so_cuoc_goi from CALL group by 1, 2 order by 3 desc
''')

97% là cuộc gọi đi từ tổng đài. Chỉ 14,5% cuộc gọi được khách nghe máy; còn lại là không nghe, máy bận, hộp thư thoại. Thời lượng chỉ lớn hơn 0 ở các cuộc ANSWER và một phần các cuộc không có kết quả hệ thống, như các cuộc gọi vào trong ví dụ ở mục 6.2; mọi kết quả khác đều có thời lượng bằng 0. Tín hiệu tương tác hợp lý từ CALL là tỷ lệ nghe máy của từng khách và tổng thời gian nói chuyện, thay vì tổng số cuộc gọi, vì số cuộc gọi phản ánh việc công ty gọi nhiều hay ít chứ không phải khách tương tác nhiều hay ít.

### 6.4. Phản hồi của khách

In [ ]:
display(q('select round(100 * avg((RESPONSE_RESULT_2 = CODE_DISPOSITION)::int), 2) pct_hai_cot_giong_nhau from CALL'))
q('''
select RESPONSE_RESULT_2, count(*) so_cuoc_goi, round(100.0 * count(*) / sum(count(*)) over (), 2) pct
from CALL group by 1 order by 2 desc limit 30
''')

`RESPONSE_RESULT_2` và `CODE_DISPOSITION` giống nhau ở 99,85% dòng, nên chỉ cần dùng một cột. 85% cuộc gọi có phản hồi "none", khớp với việc phần lớn cuộc gọi không có người nghe.

Nhiều mã bắt đầu bằng OWN hoặc chứa PTP, VoluntaryPayment, là các mã thường dùng khi gọi nhắc nợ hoặc thu hồi nợ (PTP là khách hứa trả). Như vậy bảng CALL không chỉ gồm cuộc gọi bán hàng. Câu hỏi là những khách nhận mã OWN có phải là khách đang gặp khó khăn tài chính không.

In [ ]:
q('''
with own as (select distinct SKP_CLIENT from CALL where RESPONSE_RESULT_2 like 'OWN%')
select (SKP_CLIENT in (select SKP_CLIENT from own))::int co_cuoc_goi_OWN, count(*) so_khach,
  round(100 * avg((n_active > 0)::int), 1) pct_dang_co_hd_active,
  round(100 * avg(co_qua_han), 2) pct_tung_qua_han,
  round(100 * avg(co_written_off), 2) pct_co_written_off
from CLIENT_CONTRACT group by 1
''')

Khách có cuộc gọi mã OWN có tỷ lệ đang có hợp đồng Active cao hơn hẳn (khoảng 84% so với 59%), nhưng tỷ lệ từng quá hạn gần như bằng nhóm còn lại (5,2% so với 5,1%), còn tỷ lệ Written off thấp hơn hẳn (0,09% so với 1,38%). Như vậy các cuộc gọi này chủ yếu nhắm đến khách đang trong thời gian trả nợ, không nhất thiết là khách đang trễ hạn. Nếu muốn dùng làm tín hiệu căng thẳng tài chính, chỉ nên lấy các mã cụ thể hơn như hứa trả hoặc từ chối trả, và cần kiểm chứng thêm.

### 6.5. Giờ gọi và số cuộc gọi mỗi khách

In [ ]:
display(q('''
select median(n) trung_vi_cuoc_goi_moi_khach, quantile_cont(n, 0.9) p90, max(n) cao_nhat,
       median(n_answer) trung_vi_cuoc_nghe_may, round(100 * avg((n_answer = 0)::int), 1) pct_khach_chua_nghe_lan_nao
from (select SKP_CLIENT, count(*) n, count(*) filter (where CALL_RESULT_SYSTEM = 'ANSWER') n_answer from CALL group by 1)
'''))
h = q("select hour(TIME_OF_CALL_START) gio, count(*) so_cuoc_goi, round(100 * avg((CALL_RESULT_SYSTEM = 'ANSWER')::int), 1) pct_nghe_may from CALL group by 1 order by 1")
h[(h.so_cuoc_goi > 1000)]

Một khách trung vị nhận 9 cuộc gọi nhưng chỉ nghe 1 cuộc, và 33% khách chưa nghe máy lần nào. Tỷ lệ nghe máy giảm dần trong ngày, từ khoảng 25% lúc 7–8 giờ xuống 9% lúc 15–16 giờ; khung 20 giờ chỉ có khoảng 2 nghìn cuộc nên không đáng tin. Đây có thể là gợi ý về giờ gọi cho Next Best Action, nhưng các khung giờ có thể gọi cho những nhóm khách hoặc chiến dịch khác nhau, nên chưa thể kết luận đổi giờ gọi sẽ tăng tỷ lệ nghe máy.

## 7. Mã sản phẩm giữa các bảng

Mỗi bảng dùng một bộ mã sản phẩm riêng. Muốn nối lead, sự kiện chấm điểm, cuộc gọi với hợp đồng theo sản phẩm thì phải quy đổi về cùng một bộ mã.

In [ ]:
codes = q('''
select 'CONTRACT' bang, PRODUCT ma, count(*) so_dong from CONTRACT_CLEAN group by 2
union all select 'LEADS', CODE_PRODUCT_TYPE, count(*) from LEADS_CLEAN group by 2
union all select 'PAST_BEHAVIOR', PRODUCT_CODE, count(*) from PAST_BEHAVIOR group by 2
union all select 'CALL', PRODUCT_TYPE, count(*) from CALL group by 2
''')
for b, g in codes.groupby('bang', sort=False):
    g = g.sort_values('so_dong', ascending=False)
    print(f"{b}: " + ', '.join(f"{m} ({n:,})" for m, n in zip(g.ma.fillna('trống'), g.so_dong)))

Quy đổi dự kiến về 5 nhóm của đề, dựa trên tên mã và sản phẩm đi kèm; đề chưa xác nhận nên cần ghi rõ là giả định:

- BNPL: BNPL, BNP
- CL (vay tiền mặt): CL, CLX, ACL, ACLX, ACLD
- CC (thẻ tín dụng): CC, CCX, VCC, SCW, CCW, CC_SC, CC_CC, CC_VCC
- CD và TW giữ nguyên
- Chưa rõ: CW (chỉ có ở CONTRACT), EPOS, SAI, IN

## 8. Tổng kết

Phần này gom các ý chính của notebook. Mỗi ý ghi mục chứa số liệu tương ứng ở phía trên.

**Những việc cả nhóm phải xử lý giống nhau trước khi tính điểm**

1. Khử trùng CONTRACT theo `SKP_CREDIT_CASE`, giữ dòng có thông tin cá nhân. Bảng gốc có 6,23 triệu dòng, sau khử trùng còn 4,28 triệu hợp đồng (mục 3.1).
2. Bỏ khoảng 39 nghìn dòng trùng hoàn toàn trong LEADS (mục 4.1).
3. Dùng 04/06/2025 làm ngày chốt (mục 1).
4. Xác định hợp đồng đã ký theo trạng thái, không theo ngày ký, vì hồ sơ trước 2017 không có ngày ký (mục 3.2).
5. Lấy số tiền BNPL từ `AMT_BILLING` vì `AMT_CREDIT` của BNPL bằng 0 (mục 3.3).
6. Quy đổi mã sản phẩm của 4 bảng về 5 nhóm của đề (mục 7).
7. Chọn cách dùng CALL: 75% cuộc gọi nằm sau ngày chốt và số cuộc gọi mỗi tháng gần như không đổi. `CALL_ID` bị dùng lại, mỗi dòng là một cuộc gọi (mục 6.1, 6.2).
8. Bỏ các cột gần như không có thông tin: `FLAG_CAR_OWNER`, `NAME_SALARY_FREQUENCY`, `CNT_PERSON_DEPENDENT` (mục 2).
9. Cắt ngoại lai khi dùng: thu nhập 9 tỷ đồng, khách 1.794 hồ sơ, khách 1.778 sự kiện, tuổi 1 và 123, số con trên 5 (mục 2, 3.6, 3.7, 5.1).

**Tín hiệu cho Financial Health Score**

1. Cờ trễ hạn kỳ 1–4 là tín hiệu chính: 2% hợp đồng đã ký có ít nhất một kỳ trễ, và hợp đồng Written off có 53,5% trễ ở các kỳ đầu so với 1,35% của hợp đồng Finished (mục 3.5).
2. Mức trễ hạn khác nhau theo sản phẩm, từ 1,3% ở CL đến 4,2% ở TW, nên cần so khách với mặt bằng của sản phẩm (mục 3.5).
3. Paid off có 82% trễ hạn, không nên coi là tín hiệu tốt (mục 3.5).
4. Hợp đồng nộp năm 2024–2025 trễ kỳ 1 cao gấp 2–3 lần các năm trước; cờ kỳ 4 của hợp đồng 2025 chưa đầy đủ vì nhiều hợp đồng chưa đến kỳ (mục 3.5).
5. 60% khách đang có hợp đồng Active, có thể dùng để tính gánh nặng trả nợ hiện tại (mục 3.6).
6. 182 nghìn khách (18%) chưa từng ký hợp đồng nên không có lịch sử trả nợ; cần một nhóm riêng thay vì chấm như các khách khác (mục 3.6).
7. Chênh lệch trễ hạn giữa các loại thu nhập chỉ khoảng 0,4 điểm phần trăm, nhỏ hơn nhiều so với chênh lệch giữa các sản phẩm. Đây là bằng chứng để không đưa thông tin nhân khẩu học vào điểm (mục 3.7).
8. Cuộc gọi mã OWN nhắm vào khách đang trả nợ chứ không phải khách đang trễ hạn, nên không dùng trực tiếp làm tín hiệu căng thẳng tài chính (mục 6.4).

**Tín hiệu cho Engagement Score**

1. 19% khách không có lead, sự kiện chấm điểm hay cuộc gọi nào; Engagement của nhóm này chỉ dựa được vào hợp đồng (mục 1.1).
2. Mọi khách đều nộp hồ sơ trong vòng một năm trước ngày chốt, 81,5% trong khoảng 91–180 ngày, nên recency theo hồ sơ vay gần như không phân biệt được khách; cần dùng recency của lead, sự kiện và cuộc gọi (mục 3.6).
3. Chấm điểm qua nhân viên bán hàng dẫn đến hồ sơ trong 30 ngày ở 70% trường hợp, qua app 33%, qua website đối tác 23%. Nên đặt trọng số theo kênh thay vì đếm đều mọi sự kiện (mục 5.2).
4. Chỉ 7% khách có dùng app trong năm 2025 (mục 2.3).
5. Với cuộc gọi, nên dùng tỷ lệ nghe máy và thời lượng nói chuyện thay vì số cuộc gọi, vì số cuộc gọi phản ánh việc công ty gọi nhiều hay ít. 33% khách chưa nghe máy lần nào (mục 6.3, 6.5).
6. 37% khách đã ký từ 2 sản phẩm khác nhau trở lên, là tín hiệu tương tác sản phẩm (mục 3.6).
7. Kết quả kinh doanh để kiểm chứng điểm: lead Deal có 75% nộp hồ sơ trong 30 ngày, so với 21% của lead khác. Có thể dùng Deal hoặc việc nộp hồ sơ mới làm kết quả, với điều kiện tính điểm trên giai đoạn trước và đo kết quả ở giai đoạn sau (mục 4.4).

**Gợi ý cho Next Best Action**

1. RA và SA chiếm 94% số lead nhưng tỷ lệ Deal chỉ 0,2% và 1,9%, trong khi SOB, Landing Page, TLS đạt 37–40%. Khác biệt có thể một phần do khách tự tìm đến các kênh này, nên chưa phải quan hệ nhân quả (mục 4.2).
2. 21% lead hết hạn trước khi được xử lý xong, là cơ hội cải thiện tốc độ xử lý lead (mục 4.1).
3. Tỷ lệ nghe máy giảm từ khoảng 25% lúc 7–8 giờ xuống 9% lúc 15–16 giờ (mục 6.5).